# AI basics lab 2: tool calling and agents

Run the cells top to bottom with Shift+Enter.

In [7]:
import os

from ai_basics.handmade import HANDMADE_SYSTEM, parse_tool_call
from ai_basics.llm import FakeClient, OllamaClient
from ai_basics.tools import TOOLS

# Which model to talk to. The defaults use your local Ollama.
# Set AI_BASICS_PROVIDER=fake to run everything without a model (this is what CI does).
provider = os.environ.get("AI_BASICS_PROVIDER", "ollama")
model = os.environ.get("AI_BASICS_MODEL", "qwen2.5:3b")

real_client = None if provider == "fake" else OllamaClient(model=model)


def model_or_fake(*script):
    """The real model, or (with the fake provider) a FakeClient that plays back `script`."""
    return FakeClient(list(script)) if provider == "fake" else real_client


QUESTION = "How many students per section are in CS 101?"
print(f"provider: {provider} | model: {model}")

provider: ollama | model: qwen2.5:3b


## Part A. Tool calling by hand

In [8]:
print(HANDMADE_SYSTEM)
print("-" * 60)

# With the fake provider, this is the reply the fake plays back. A real model ignores it.
client = model_or_fake('{"tool": "get_course_info", "arguments": {"course": "CS 101"}}')
MESSAGES = [
    {"role": "system", "content": HANDMADE_SYSTEM},
    {"role": "user", "content": QUESTION},
]

reply = client.chat(MESSAGES).text
print("raw reply:", repr(reply))

call = parse_tool_call(reply)
print("parsed:   ", call)
if call:
    # The model only wrote a string. This line is where YOUR code runs the function.
    print("result:   ", TOOLS[call.name](**call.arguments))

You can use two tools.

Tool: get_course_info
  Argument: course (text, like "CS 101")
  Returns: the course title, number of students and number of sections.

Tool: calculate
  Argument: expression (text, like "84 / 3")
  Returns: the result. Supports + - * / and parentheses. Use it for every calculation.

To use a tool, reply with ONLY this JSON and nothing else:
{"tool": "<tool name>", "arguments": {"<argument name>": "<value>"}}

When you have the final answer, reply with ONLY this JSON and nothing else:
{"answer": "<your answer>"}

Never write any text outside the JSON.
------------------------------------------------------------
raw reply: '{"tool": "get_course_info", "arguments": {"course": "CS 101"}}'
parsed:    ToolCall(name='get_course_info', arguments={'course': 'CS 101'})
result:    {'course': 'CS 101', 'title': 'Intro to Programming', 'students': 84, 'sections': 3}


In [9]:
# Same question 5 times at temperature 1.0. seed=i makes each try repeatable.
# The fake's script: 3 good replies, then a chatty one and a guessed answer.
client = model_or_fake(
    '{"tool": "get_course_info", "arguments": {"course": "CS 101"}}',
    '```json\n{"tool": "get_course_info", "arguments": {"course": "CS 101"}}\n```',
    '{"tool": "get_course_info", "arguments": {"course": "cs101"}}',
    'Sure! {"tool": "get_course_info", "arguments": {"course": "CS 101"}}',
    '{"answer": "28"}',
)

parsed = 0
for i in range(5):
    reply = client.chat(MESSAGES, temperature=1.0, seed=i).text
    ok = parse_tool_call(reply) is not None
    parsed += ok
    print(f"{'OK  ' if ok else 'FAIL'}  {reply[:70]!r}")
print(f"parsed {parsed} of 5")

OK    '{"tool": "get_course_info", "arguments": {"course": "CS 101"}}'
OK    '{"tool": "get_course_info", "arguments": {"course": "CS 101"}}'
OK    '{"tool": "get_course_info", "arguments": {"course": "CS 101"}}'
OK    '{"tool": "get_course_info", "arguments": {"course": "CS 101"}}'
OK    '{"tool": "get_course_info", "arguments": {"course": "CS 101"}}'
parsed 5 of 5


**What I learned (Part A):**

- 

## Part B. The tool API

In [10]:
from ai_basics.llm import ToolCall
from ai_basics.tools import TOOL_SCHEMAS

USER_ONLY = [{"role": "user", "content": QUESTION}]

# With the fake provider, these two scripted replies play back. A real model ignores them.
client = model_or_fake(
    "I don't have information about CS 101's enrollment.",
    [ToolCall("get_course_info", {"course": "CS 101"})],
)
plain = client.chat(USER_ONLY)
with_tools = client.chat(USER_ONLY, tools=TOOL_SCHEMAS)

# The fake reports 100 input tokens every time, so with the fake this shows 0.
# Only a real model measures the cost.
print("no tools  :", repr(plain.text), plain.tool_calls, f"| input tokens: {plain.input_tokens}")
print(
    "with tools:",
    repr(with_tools.text),
    with_tools.tool_calls,
    f"| input tokens: {with_tools.input_tokens}",
)
print(f"the tool descriptions cost {with_tools.input_tokens - plain.input_tokens} tokens")

no tools  : "I'm sorry, but I don't have enough information to answer your question. The number of students per section in CS 101 varies depending on the institution, the specific course offering, and the semester. To provide an accurate answer, I would need to know which institution you are referring to and the particular semester and section you are interested in. Could you please provide more details?" [] | input tokens: 42
with tools: '' [ToolCall(name='get_course_info', arguments={'course': 'CS 101'})] | input tokens: 265
the tool descriptions cost 223 tokens


**What I learned (Part B):**

- 

## Part C. The agent loop

In [11]:
from ai_basics.agent import run_agent
from ai_basics.llm import ToolCall

# With the fake provider, this script plays back: lookup, calculate, then the answer.
# A real model ignores it and makes its own choices.
llm = model_or_fake(
    [ToolCall("get_course_info", {"course": "CS 101"})],
    [ToolCall("calculate", {"expression": "84 / 3"})],
    "CS 101 has 28 students per section.",
)
result = run_agent(llm, QUESTION, verbose=True)

print()
print("answer:            ", result.answer)
print("stopped:           ", result.stopped)
print("model calls:       ", result.steps)
print("total input tokens:", result.input_tokens)

step 1 (284 input tokens)
  [assistant wants] get_course_info({"course": "CS 101"})
  [tool → get_course_info] {"course": "CS 101", "title": "Intro to Programming", "students": 84, "sections": 3}
step 2 (356 input tokens)
  [assistant] There are 28 students per section in CS 101.

answer:             There are 28 students per section in CS 101.
stopped:            answer
model calls:        2
total input tokens: 640


In [12]:
# The whole history, exactly as the model saw it at its last call.
for i, message in enumerate(result.messages):
    print(i, message)

0 {'role': 'system', 'content': 'You are a course assistant. Use the tools to look up facts and to do arithmetic; never guess a number. When you have the answer, reply in one short sentence.'}
1 {'role': 'user', 'content': 'How many students per section are in CS 101?'}
2 {'role': 'assistant', 'content': '', 'tool_calls': [{'function': {'name': 'get_course_info', 'arguments': {'course': 'CS 101'}}}]}
3 {'role': 'tool', 'tool_name': 'get_course_info', 'content': '{"course": "CS 101", "title": "Intro to Programming", "students": 84, "sections": 3}'}
4 {'role': 'assistant', 'content': 'There are 28 students per section in CS 101.'}


**What I learned (Part C):**

- 

## Part D. Stopping conditions

In [13]:
from ai_basics.agent import run_agent
from ai_basics.llm import ToolCall

# D1: the step limit. This question needs two model calls, so one is not enough.
llm = model_or_fake([ToolCall("get_course_info", {"course": "CS 101"})])
result = run_agent(llm, QUESTION, max_steps=1, verbose=True)

print()
print("stopped:", result.stopped)
print("answer: ", result.answer)

step 1 (284 input tokens)
  [assistant wants] get_course_info({"course": "CS 101"})
  [tool → get_course_info] {"course": "CS 101", "title": "Intro to Programming", "students": 84, "sections": 3}

stopped: max_steps
answer:  I hit the step limit (1) before finding an answer.


In [14]:
from ai_basics.agent import run_agent
from ai_basics.llm import ToolCall

# D2: a tool error. The lookup fails; the error text goes back to the model as a tool message.
llm = model_or_fake(
    [ToolCall("get_course_info", {"course": "CS 999"})],
    "I couldn't find CS 999. The known courses are CS 101, CS 230 and CS 350.",
)
result = run_agent(llm, "How many students per section are in CS 999?", verbose=True)

print()
print("stopped:", result.stopped)
print("answer: ", result.answer)

step 1 (284 input tokens)
  [assistant wants] get_course_info({"course": "CS 999"})
  [tool → get_course_info] error: LookupError: unknown course 'CS 999'; known courses: CS 101, CS 230, CS 350
step 2 (360 input tokens)
  [assistant] The course CS 999 is not found in the database. Please check the course code and try again.

stopped: answer
answer:  The course CS 999 is not found in the database. Please check the course code and try again.


In [15]:
from ai_basics.agent import run_agent

# D3: no tool needed. Does the model answer straight away, or reach for a tool anyway?
llm = model_or_fake("CPU stands for Central Processing Unit.")
result = run_agent(llm, "What does CPU stand for?", verbose=True)

print()
print("stopped:", result.stopped)
print("answer: ", result.answer)

step 1 (277 input tokens)
  [assistant] CPU stands for Central Processing Unit.

stopped: answer
answer:  CPU stands for Central Processing Unit.


In [16]:
from ai_basics.agent import run_agent
from ai_basics.llm import ToolCall
from ai_basics.tools import TOOL_SCHEMAS, TOOLS, calculate

# D4: a job that needs several lookups and then arithmetic, with and without the calculator.
TEXTBOOK = (
    "Every student in CS 101, CS 230 and CS 350 buys a $37 textbook. How much is spent in total?"
)
# One reply with three tool calls: the model asks for all three lookups at once.
THREE_LOOKUPS = [
    ToolCall("get_course_info", {"course": "CS 101"}),
    ToolCall("get_course_info", {"course": "CS 230"}),
    ToolCall("get_course_info", {"course": "CS 350"}),
]

print("=== with all tools ===")
llm = model_or_fake(
    THREE_LOOKUPS,
    [ToolCall("calculate", {"expression": "(84 + 45 + 36) * 37"})],
    "In total $6,105 is spent on textbooks.",
)
with_calc = run_agent(llm, TEXTBOOK, max_steps=8, verbose=True)

print()
print("=== with get_course_info only ===")
lookup_only = {"get_course_info": TOOLS["get_course_info"]}
lookup_schemas = [s for s in TOOL_SCHEMAS if s["function"]["name"] == "get_course_info"]
# Without calculate, a real model has to work out 165 * 37 itself.
llm = model_or_fake(THREE_LOOKUPS, "In total $6,105 is spent on textbooks.")
no_calc = run_agent(
    llm, TEXTBOOK, max_steps=8, verbose=True, tools=lookup_only, schemas=lookup_schemas
)

print()
print("with all tools:      ", with_calc.stopped, "|", with_calc.answer)
print("get_course_info only:", no_calc.stopped, "|", no_calc.answer)
print("true total:          ", f"{calculate('(84 + 45 + 36) * 37'):,.0f}")

=== with all tools ===
step 1 (305 input tokens)
  [assistant wants] get_course_info({"course": "CS 101"})
  [assistant wants] get_course_info({"course": "CS 230"})
  [assistant wants] get_course_info({"course": "CS 350"})
  [tool → get_course_info] {"course": "CS 101", "title": "Intro to Programming", "students": 84, "sections": 3}
  [tool → get_course_info] {"course": "CS 230", "title": "Data Structures", "students": 45, "sections": 2}
  [tool → get_course_info] {"course": "CS 350", "title": "Operating Systems", "students": 36, "sections": 1}
step 2 (503 input tokens)
  [assistant] 

=== with get_course_info only ===
step 1 (238 input tokens)
  [assistant wants] get_course_info({"course": "CS 101"})
  [assistant wants] get_course_info({"course": "CS 230"})
  [assistant wants] get_course_info({"course": "CS 350"})
  [tool → get_course_info] {"course": "CS 101", "title": "Intro to Programming", "students": 84, "sections": 3}
  [tool → get_course_info] {"course": "CS 230", "title": "Dat

**What I learned (Part D):**

- 